# MEVIS – Machine-learning guided Experimental in Vivo Improvement of Systems

This notebook implements the MEVIS (Machine-learning guided Experimental in Vivo Improvement of Systems) workflow used in *Schulz-Mirbach et al.* to optimize expression of a synthetic methylsuccinate-assimilation module in *E. coli*. MEVIS is an adaptation of the METIS active-learning framework of Pandi *et al.* (Nat Commun, 2022; https://github.com/amirpandi/METIS) to *in vivo* pathway optimization, in which inducer concentrations, carbon-source concentrations, and selection-substrate concentrations are jointly varied between rounds and growth (OD<sub>600</sub>·µ<sub>max</sub>) is used as the objective function.

## What this notebook does
1. Defines design variables (inducers / substrates) and their ranges.
2. Generates Hammersley-distributed input combinations for Round 1.
3. After each experimental round, ingests a `Results_n.csv` file containing the measured objective, retrains an ensemble of XGBoost regressors, and proposes the next set of input combinations using an Upper Confidence Bound (UCB) acquisition function.
4. Produces all auxiliary files required to run the experiments on an ECHO acoustic liquid handler.
5. Provides diagnostic visualizations: per-round objective distributions, per-variable scatter plots, sampled-condition trajectories, feature importance, pairwise non-linear interactions, and informative-sample selection.

## Expected file layout
The notebook is intended to be run from a working directory that contains `utils.py` and, after each round, a `Results_n.csv` (n = 1, 2, ...) with the experimental measurements for the combinations generated in `Day_n/Concentrations_n.csv`. See the accompanying README for a worked example.


## 1. Imports


In [1]:
import pandas as pd
import numpy as np
import os
#!pip install xgboost
from xgboost import XGBRegressor
import time
from sklearn.model_selection import RandomizedSearchCV
from itertools import product
import seaborn as sns
import matplotlib.pyplot as plt
from collections.abc import Iterable
import math
import random
from scipy.stats import spearmanr, pearsonr
#!pip install scikit-optimize  # see README for install instructions


pd.options.mode.chained_assignment = None  # default='warn'

# Load helper functions from utils.py located in the same folder as this notebook
from utils import *

## 2. User inputs

Fill out the cells below before each MEVIS run. All units are nanoliters for volumes and the unit of the user's choice (e.g. µM, mM) for concentrations — but units must be **consistent** across `Conc_Min`, `Conc_Max`, `Conc_Values` and `Conc_Stock` for each variable.


### Workflow overview

Before each experimental round:

1. Set the global experimental parameters (Section 2.1).
2. Define the variable ranges and stock concentrations (Section 2.2).
3. Add any control / reference combinations that should always be included (Section 2.3).

After each experimental round, save the measured objective values in `Results_n.csv` in the working directory and run the cells under "Subsequent rounds" to generate the next batch of input combinations.


### 2.1 Global experimental parameters


In [ ]:
# Global experimental parameters
m = 96                              # Number of combinations per round. Must not exceed the usable wells of the destination plate.
minimum_drop_size_nanoliter = 2.5   # Minimum droplet volume the ECHO can dispense reliably.
final_reaction_volume_nanoliter = 150000  # Total well volume (here: 10 µL of ECHO-dispensed mix + 140 µL of M9/cell suspension).
maximum_volume_of_model_output = 35000    # Total volume that MEVIS is allowed to allocate (= final volume minus fixed components, with a margin to avoid water = 0).
fixed_parts = {"M9, 2KG, Glycerol, Cell Mixture": 0.7666}  # Components added at a fixed fraction of the well volume (here: 115/150 µL = 76.6%).
Master_Mix_for_fixed_parts = True   # If True, the fixed parts are pipetted manually as a master mix and excluded from the ECHO transfer file.
days_total = 5                      # Maximum number of MEVIS rounds the run will execute.

# Model hyperparameters
RandomCV = True
n_iter = 200                        # Number of hyperparameter combinations sampled during RandomizedSearchCV.
ensemble_len = 20                   # Size of the XGBoost ensemble used for UCB acquisition.
exploration = 0.5                   # UCB exploration weight (the exploitation weight is set to 1 in the call below).
days_range = [m for i in range(days_total)]

### 2.2 Define design variables and stock concentrations

For each design variable (inducer, carbon source, selection substrate, etc.), specify:

- `Conc_Min`, `Conc_Max`: lower/upper bounds (set `Conc_Values=None` to let MEVIS sample continuously between these limits).
- `Conc_Values`: a list of allowed discrete concentrations (set `Conc_Min=Conc_Max=None` if used).
- `Conc_Stock`: stock concentration; can be a single value or a list (multi-stock setups).
- `Alternatives`: list of mutually-exclusive variants (e.g. different buffers); set to `None` if not used.

**Important:** variable names must not contain underscores (`_`), as the downstream code uses `_` as a delimiter for alternatives and stock types.


In [ ]:
# Example concentrations_limits used in the methylsuccinate-MEVIS run reported in the manuscript.
# Conc_Min / Conc_Max in nM; Conc_Stock in nM.
concentrations_limits = {
    'Cumate':          {'Conc_Min': 0, 'Conc_Max':   300, 'Conc_Values': None, 'Conc_Stock':  100000, 'Alternatives': None},
    'Methylsuccinate': {'Conc_Min': 0, 'Conc_Max': 20000, 'Conc_Values': None, 'Conc_Stock':  200000, 'Alternatives': None},
    'DAPG':            {'Conc_Min': 0, 'Conc_Max':   500, 'Conc_Values': None, 'Conc_Stock':   25000, 'Alternatives': None},
    'Glycerol':        {'Conc_Min': 0, 'Conc_Max': 20000, 'Conc_Values': None, 'Conc_Stock': 2000000, 'Alternatives': None},
    '2-KG':            {'Conc_Min': 0, 'Conc_Max': 20000, 'Conc_Values': None, 'Conc_Stock':  500000, 'Alternatives': None},
}

### 2.2.1 Sanity-check the sampleable grid

This cell prints the possible concentration values for each variable that satisfy the droplet-size constraint of the ECHO. Use the printed values to refine `Conc_Stock` or switch to a discrete `Conc_Values` list if the grid is too coarse.


In [ ]:
# Check Possible Concentrations
data_lists = {}
num = 0

pool_size = 1

for key, value in concentrations_limits.items():
    print('Possible Conc For :',key)
    if not value['Conc_Values']:
        print('Your Min, Max :', "({}, {})".format(value['Conc_Min'], value['Conc_Max']))
    concs, vols = allowed_output(value, reaction_vol_nl = final_reaction_volume_nanoliter, drop_size_nl = minimum_drop_size_nanoliter, verbose=0)
    print(concs)
    print(vols)

    pool_size *= len(concs)

    data_lists[num] = vols
    num += 1
    print()

print('All Possible Combination Number = ', pool_size)
if pool_size > 10000000:
    pool_size = 10000000
    print('Percentage calculation is not availbe duo to large pool size!')
else:
    percent , pool_size = percentage_possible(data_lists, threshold = maximum_volume_of_model_output)
    print(percent, f'% of {pool_size} possible combination are executable!')

### 2.3 Reference and control combinations

Any combination listed in `specials` will be added to every round. Leave the dictionary empty if no references are needed (the cell still has to be executed).


In [ ]:
# Reference / negative-control combinations. Concentrations must be compatible with the minimum drop size.
# See the original METIS repository (https://github.com/amirpandi/METIS) for examples of how to populate this dictionary.
specials = {
}

## 3. Round 1 — initial Hammersley sampling

Run the cells in this section once at the beginning of a MEVIS campaign, before any experimental data have been collected.


### 3.1 Generate the initial Hammersley sample

`hammersley_initial_generator` draws a quasi-random, low-discrepancy sample from the design space (improving on uniform random sampling when only a small number of initial points can be evaluated).


In [ ]:
# make random combinations
Concentrations_1 = hammersley_initial_generator(concentrations_limits, number_of_combination = m,
                                                reaction_vol_nl=final_reaction_volume_nanoliter,
                                                max_nl=maximum_volume_of_model_output, drop_size_nl=minimum_drop_size_nanoliter, return_df=True)

# add control, reference and other desired combinations
df_specials = [pd.DataFrame(i) for i in specials.values()]
Concentrations_1 = pd.concat([Concentrations_1, *df_specials]).reset_index(drop=True)

Concentrations_1

### 3.2 Save the Round 1 concentration file


In [ ]:
!mkdir -p Day_1
Concentrations_1.to_csv('Day_1/Concentrations_1.csv', index=False)

### 3.3 Convert concentrations to ECHO transfer volumes


In [ ]:
# concentration_to_volume
Volumes_1 = concentration_to_volume(Concentrations_1, concentrations_limits, reaction_mixture_vol_nl=final_reaction_volume_nanoliter,
                                    fixed_parts=fixed_parts, minimum_drop_size_nanoliter=minimum_drop_size_nanoliter)
Volumes_1.to_csv('Day_1/Volumes_1.csv', index=False)
Volumes_1

## 4. Subsequent rounds — active learning loop

Run this section after each round once `Results_n.csv` has been saved. The notebook automatically detects the most recent completed round and proposes the next batch.


### 4.1 Detect the most recent completed round


In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]
print(start_day, day)

### 4.2 Aggregate experimental data from all previous rounds

Builds the training table that will be fed to the ensemble regressors.


In [ ]:
# this cell generates a table with all experimental data that have been collected thus far.
desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day + 1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

if 'reference' in specials.keys():
    ref_data = pd.DataFrame(specials['reference'])
    ref_label = pd.DataFrame({'yield':[1.0]})
    aggregated_data_m = pd.concat([aggregated_data_m, ref_data[desired_cols]]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, ref_label]).reset_index(drop=True)

aggregated_data_m

### 4.3 Train the XGBoost ensemble

An ensemble of `ensemble_len` XGBoost regressors is created. With `RandomCV = True`, the top hyperparameter combinations identified by 5-fold RandomizedSearchCV are used to initialise the ensemble; this provides a non-Gaussian estimate of the predictive variance that the UCB acquisition function (Section 4.4) uses to drive exploration.

The hyperparameters scanned are:

- `learning_rate`: step size for boosting updates.
- `colsample_bytree`: fraction of features used per tree (regularisation).
- `subsample`: fraction of samples used to grow each tree (regularisation).
- `max_depth`: tree depth (model complexity).
- `n_estimators`: number of boosting rounds.
- `reg_lambda`: L2 regularisation on weights.
- `gamma`: minimum loss reduction for further partitioning.
- `min_child_weight`: minimum instance weight in a child node.


In [ ]:
# ensemble of regressors
# XGBRegressor is an enhanced random forest boosted algorithm
RandomCV = True
if RandomCV:
    # create a default XGBoost classifier
    model = XGBRegressor(objective = 'reg:squarederror')

    # Create the grid search parameter and scoring functions:
    # learning_rate: The step size for updating the weights, controlling the contribution of each tree.
    # colsample_bytree: The fraction of features to be used for each tree, aiding in preventing overfitting.
    # subsample: The fraction of samples used to grow each tree, helping reduce overfitting.
    # max_depth: The maximum depth of a tree, determining the complexity of the model.
    # n_estimators: The number of boosting rounds (trees) to be built, affecting accuracy and training time.
    # reg_lambda: L2 regularization term on weights, penalizing large weights to prevent overfitting.
    # gamma: Minimum loss reduction needed for further partitioning a leaf node, making the model more conservative.
    # min_child_weight: Minimum sum of instance weights needed in a child node, controlling overfitting by setting a threshold for partitioning.
    param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

    # create the grid search object
    grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

    print('RandomSearchCV ...')

    # try different hyperparameter combinations and sort based on cross validation performance
    grid.fit(aggregated_data_m.values, aggregated_label_m.values)
    results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)

    # Ensemble Creation: Uses the top-performing hyperparameter combinations to create an ensemble of XGBoost regressors, each initialized with a different set of optimal hyperparameters.
    regressors_list = [XGBRegressor(
                       objective = 'reg:squarederror',**param) for param in results.params.iloc[0:ensemble_len,] ]
    print('RandomSearchCV Done!')

else:
    # Default: if RandomCV is false just use fixed hyperparameters.
    regressors_list = [XGBRegressor(
        objective = 'reg:squarederror',
        n_estimators = n,
        learning_rate = l ,
        max_depth = 6,
        min_child_weight = 4,
        subsample = 0.9,
        gamma = 0.4,
        colsample_bytree = 0.9) for n in (10, 20, 30, 40, 50) for l in (0.01, 0.03, 0.1, 0.2)]

### 4.4 Pool size for the acquisition function


In [ ]:
pool_size
# pool_size: how many random combinations to ask the regressors to predict in each round

### 4.5 Propose the next batch with Upper Confidence Bound (UCB) acquisition

The trained ensemble is used to score `pool_size` randomly drawn candidate combinations. Each candidate is ranked by
$$\mathrm{UCB} = \text{exploitation} \cdot \bar{y} + \text{exploration} \cdot \sigma_y,$$
where $\bar{y}$ is the ensemble mean and $\sigma_y$ the ensemble standard deviation. The top `m` non-repeated combinations are returned for the next experimental round.


In [ ]:
t0 = time.time()

# pool_size: how many random combinations to ask the regressors to predict in each round
pool_size = 10000 # if you want to change pool size, default = all possible combinations

Concentrations_n_m = bayesian_optimization(regressors_list, aggregated_data_m, aggregated_label_m, concentrations_limits,
                                           final_order=final_order,
                                           df_main = aggregated_data_m,
                                           reaction_vol_nl=final_reaction_volume_nanoliter, max_nl=maximum_volume_of_model_output,
                                           drop_size_nl=minimum_drop_size_nanoliter,
                                           exploitation=1, exploration=0.5, test_size=m, pool_size=pool_size, verbose=0,
                                           day=day, days_range = days_range)


# regressors_list: The ensemble of XGBoost regressors trained and optimized earlier.
# aggregated_data_m: The DataFrame containing the aggregated experimental data.
# aggregated_label_m: The DataFrame containing the corresponding yield values.
# concentrations_limits: The dictionary specifying the concentration limits and alternatives for each factor.
# final_order: The final order of columns in the dataset, which ensures consistency in data structure.
# df_main: Again, the aggregated DataFrame with experimental data (repeated for clarity in the function call).
# reaction_vol_nl: The final reaction volume in nanoliters, a critical parameter for the experimental setup.
# max_nl: The maximum volume of the model output in nanoliters.
# drop_size_nl: The minimum drop size in nanoliters, important for precision in liquid handling.
# exploitation: A parameter controlling the exploitation aspect of the Bayesian optimization (typically set to 1).
# exploration: Controls the exploration aspect, retrieved from a predefined list exploration for the current day.
# test_size: the number of utput combinations. The size of the test set (parameter m), which determines how much data is used for testing the model's performance.
# pool_size: The size of the pool of combinations to consider during the optimization, set to 10,000.
# verbose: Verbosity level of the function's output, set to 0 to suppress detailed output.
# day: The current day in the experimental timeline.
# days_range: The range of days over which the optimization is performed.

print("Passed Time(s): ",time.time()-t0)

Concentrations_n_m

### 4.6 Append specials and save the next-round concentration file


In [ ]:
# add control, reference and other desired combinations
df_specials = [pd.DataFrame(i) for i in specials.values()]
Concentrations_n = pd.concat([Concentrations_n_m, *df_specials]).reset_index(drop=True)

name_folder = 'Day_{}'.format(day+1)
! mkdir -p {name_folder}

Concentrations_n.to_csv('Day_{}/Concentrations_{}.csv'.format(day+1, day+1), index=False)

### 4.7 Sanity-check that no combinations are repeated across rounds


In [ ]:
# check not to make repeated combination
df_main = aggregated_data_m

comparison_df = df_main.merge(pd.read_csv('Day_{}/Concentrations_{}.csv'.format(day+1, day+1)).iloc[:m,:],
                              indicator=True,
                              how='outer')

comparison_df._merge.unique()

### 4.8 Convert the next-round concentrations to ECHO transfer volumes


In [ ]:
# concentration_to_volume
Volumes_n = concentration_to_volume(Concentrations_n, concentrations_limits, reaction_mixture_vol_nl=final_reaction_volume_nanoliter, fixed_parts=fixed_parts, minimum_drop_size_nanoliter=minimum_drop_size_nanoliter)

Volumes_n.to_csv('Day_{}/Volumes_{}.csv'.format(day+1, day+1), index=False)

## 5. Diagnostics — per-round objective distribution


In [ ]:
display_std = False # if you have included "yield_std" column in Results.csv will show it else disply 0 for std
day = day_finder('Results')
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

Results_m = pd.DataFrame(columns=['yield', 'std', 'Day'])
for i in range(start_day, day):
    Results_i = pd.DataFrame(columns=['yield', 'std', 'Day'])
    if start_day:
        Results_i['yield'] = pd.read_csv(f'Results_{i}.csv')['yield'].iloc[0:days_range[i-1]]
    else:
        Results_i['yield'] = pd.read_csv(f'Results_{i}.csv')['yield'].iloc[0:len(pd.read_csv('Results_0.csv'))]

    try:
        if start_day:
            Results_i['std'] = pd.read_csv(f'Results_{i}.csv')['yield_std'].iloc[0:days_range[i-1]]
        else:
            Results_i['std'] = pd.read_csv(f'Results_{i}.csv')['yield_std'].iloc[0:len(pd.read_csv('Results_0.csv'))]

    except:
        Results_i['std'] = 0
    Results_i['Day'] = f'Day_{i}'
    Results_m = pd.concat([Results_m, Results_i])

#plt.style.use('seaborn-whitegrid')
#plt.style.use('seaborn-poster')
fig, ax = plt.subplots(1, 1)

ax = sns.swarmplot(x='Day', y='yield', data=Results_m, color='#00A368', alpha=0.9, size=8)
#ax = sns.boxplot(x='Day', y='yield', data=Results_m, color='white', fliersize=0, linewidth=4)


if display_std:
    # Find the x,y coordinates for each point
    x_coords = []
    y_coords = []
    for point_pair in ax.collections:
        for x, y in point_pair.get_offsets():
            x_coords.append(x)
            y_coords.append(y)

    errors = Results_m['std']
    ax.errorbar(x_coords, y_coords, yerr=errors, fmt=' ',color="grey", zorder=3, linewidth=1.2)
    print(errors, x_coords, y_coords)

ax

In [ ]:
#fig.savefig(f'Day_{day-1}_Boxplot.png', format='png', dpi=1200)
fig.savefig(f'Day_4_Boxplot.svg', format='svg', dpi=1200)

## 6. Diagnostics — objective vs. each design variable


In [ ]:
desired_cols = []
day = day_finder('Results')
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day):
    if start_day:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

#plt.style.use('seaborn-whitegrid')

num = len(desired_cols)
dim = math.ceil(math.sqrt(num))
data = pd.concat([aggregated_data_m, aggregated_label_m], axis=1)

import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 4.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally

fig = plt.figure(figsize=(20,30))

for i in range(1, num+1):
    ax = plt.subplot(dim, dim, i)
    ax.xaxis.grid(False)
    ax = sns.scatterplot(x = desired_cols[i-1], y='yield', color='#3944BC', data=data, s=50, linewidth=0, alpha=0.3)
    y_axis = ax.axes.get_yaxis()
    y_axis.set_label_text('foo')
    y_label = y_axis.get_label()
    ##print isinstance(x_label, matplotlib.artist.Artist)
    y_label.set_visible(False)
    #ax.xaxis.set_tick_params()
    plt.xticks(weight = 'bold', fontsize=18)
    plt.yticks(weight = 'bold', fontsize=18)

    plt.xlabel(desired_cols[i-1], labelpad=30 ,fontsize=25, weight='bold')


fig.tight_layout(h_pad=3, w_pad=3)

In [ ]:
#fig.savefig(f'Day_{day-1}_Metabolite_Yield.png', format='png', dpi=1200)
fig.savefig(f'Day_4_Metabolite_Yield.svg', format='svg', dpi=1200)

## 7. Diagnostics — sampled-condition trajectories across rounds


In [ ]:
data_all = pd.DataFrame(columns = list(concentrations_limits.keys())+['day'])
day = day_finder('Results')
desired_cols = []

for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

d = (1, 0)[os.path.isfile('Results_0.csv')]

while os.path.exists(f'Results_{d}.csv'):
    data = pd.read_csv(f'Results_{d}.csv')
    data = data[[*list(concentrations_limits.keys())]]
    data = data.iloc[:days_range[d-1],:]
    data['day'] = f'Day_{d}'
    data_all = pd.concat([data_all, data]).reset_index(drop=True)
    d += 1

import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 4.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 10.0 #set the value globally


num = len(concentrations_limits)
dim = math.ceil(math.sqrt(num))

fig = plt.figure(figsize=(20,30))
for i in range(1, num+1):
    ax = plt.subplot(dim, dim, i)
    ax = sns.stripplot(x=list(concentrations_limits.keys())[i-1], y="day", data=data_all, color='#3944BC', s=10, linewidth=0, alpha=0.5)
    y_axis = ax.axes.get_yaxis()
    y_axis.set_label_text('foo')
    y_label = y_axis.get_label()
    y_label.set_visible(False)

    ax.yaxis.set_ticklabels([])
    plt.xticks(weight = 'bold', fontsize=18)

    plt.xlabel(desired_cols[i-1], labelpad=30 ,fontsize=25, weight='bold')


fig.tight_layout(h_pad=3, w_pad=3)

In [ ]:
fig.savefig(f'Day_{day-1}_Metabolite_Days.png', format='png', dpi=1200)
fig.savefig(f'Day_{day-1}_Metabolite_Days.svg', format='svg', dpi=1200)

## 8. Export — Table2Speech-compatible volume file

Optional: converts the per-round `Volumes_n.csv` file into a Table2Speech-readable layout (https://gitlab.com/dna-and-natural-algorithms-group/table2speech).


In [ ]:
which_day = 1  # Round whose Volumes_<which_day>.csv should be converted (file must be located in Day_<which_day>/).

In [ ]:
data = pd.read_csv(f'Day_{which_day}/Volumes_{which_day}.csv')

final_columns = []
columns_name = data.columns
for column in columns_name:
    if "_" in column:
        column_value = column.split('_')[0]
        column_to_add = data[column].unique()
        df_to_add = pd.DataFrame(columns = column_to_add,  data=np.zeros((len(data),len(column_to_add))))
        for i in range(len(data)):
            df_to_add[data[column].iloc[i,]][i] = data[column_value].iloc[i,]
        data = pd.concat([data, df_to_add], axis=1)
        final_columns.remove(column_value)
        final_columns += list(column_to_add)
    else:
        final_columns.append(column)

data = data[final_columns]
data

In [ ]:
data.to_csv(f'Table2Speech_Volumes_{which_day}.csv', index=False)

## 9. Export — ECHO acoustic-liquid-handler transfer files

Generates the `Echo_Input_day<n>.csv` file directly usable with the ECHO liquid handler, plus a human-readable pipetting instruction for the source plate.


In [ ]:
# Experimental settings for the ECHO export
which_day = 1                       # Round whose Volumes_<which_day>.csv should be exported.
triplicate = False                  # If True, three technical replicates per combination are generated.
vertical = True                     # Fill the destination plate column-by-column. Horizontal fill yields fewer combinations per plate.
max_volume_per_well = 30000         # Maximum usable volume per source well (nL).
Master_Mix_for_fixed_parts = True   # If True, fixed parts are added as a master mix and excluded from the ECHO file.

# Destination plate settings
destination_plate_384_well = False  # False = 96-well destination plate.
destination_start_well = "A1"       # Starting well of the destination plate ('B2' is recommended for 384-well plates to skip edge wells).

# Source plate settings (384-well plate, reused across rounds)
source_start_well = "A1"

In [ ]:
# Load the round's Volumes file and reshape any "alternative" columns into one-hot encoded columns
# so that each alternative can be assigned its own ECHO source well.
data = pd.read_csv(f'Day_{which_day}/Volumes_{which_day}.csv')

final_columns = []
columns_name = data.columns

for column in columns_name:
    if "_" in column:
        column_value = column.split('_')[0]
        column_to_add = data[column].unique()
        df_to_add = pd.DataFrame(columns=column_to_add, data=np.zeros((len(data), len(column_to_add))))
        for i in range(len(data)):
            df_to_add[data[column].iloc[i,]][i] = data[column_value].iloc[i,]
        data = pd.concat([data, df_to_add], axis=1)
        final_columns.remove(column_value)
        final_columns += list(column_to_add)
    else:
        final_columns.append(column)

# Remove fixed parts (handled outside ECHO when Master_Mix_for_fixed_parts is True).
for part in fixed_parts.keys():
    if part in final_columns:
        final_columns.remove(part)

data_ECHO = data[final_columns]
data_ECHO

### 9.1 Validate destination plate capacity


In [ ]:
# This function checks if the correct well plate was chosen.
check_wells_capacity(volumes_array=data_ECHO, triplicate=triplicate, destination_plate_384_well=destination_plate_384_well, vertical=vertical)

### 9.2 Assign source wells


In [ ]:
# Assign each item to source wells.
source_wells_df = calculate_source_wells(data_ECHO, triplicate=triplicate, source_start_well=source_start_well, max_volume_per_well=max_volume_per_well)
source_wells_df.to_csv(f'Source_wells_{which_day}.csv', index=False)

source_wells_df

### 9.3 Assign destination wells


In [ ]:
# Assign each combination to a destination well.
# Each triplicate is separated by a column.
destination_df = put_volumes_to_wells(data_ECHO, destination_plate_384_well=destination_plate_384_well, vertical=vertical, triplicate=triplicate, make_csv=False)

destination_df

### 9.4 Build the ECHO transfer table


In [ ]:
# Assigns each destination well to source wells for each transfer of items.
def create_ECHO_transfer_table(destination_df, source_wells_df, final_reaction_volume_nanoliter, minimum_drop_size_nanoliter, fixed_parts=None, Master_Mix_for_fixed_parts=False):
    """
    Create a transfer table for ECHO machine from source wells to destination wells.

    Parameters
    ----------
    destination_df: DataFrame
        DataFrame that includes destination wells with volumes (output from put_volumes_to_wells).

    source_wells_df: DataFrame
        DataFrame that includes source wells with volumes (output from calculate_source_wells).

    minimum_drop_size_nanoliter: int
        Minimum drop size that can be transferred.

    fixed_parts: dict
        Dictionary of fixed parts with their corresponding proportions.

    Master_Mix_for_fixed_parts: bool
        If True, delete rows corresponding to items in fixed_parts and summarize their total volumes.

    Returns
    -------
    ECHO_transfer_table_df: DataFrame
        DataFrame that includes item, source well, destination well, and transfer volume.

    fixed_parts_volumes: dict
        Dictionary summarizing the volumes of each fixed part to be added to each destination well.
    """
    # Initialize the transfer table
    transfer_table = []
    fixed_parts_volumes = {item: 0 for item in fixed_parts.keys()} if fixed_parts else {}

    # Iterate through each item.
    for item in destination_df.columns:
        if item == 'well_name':
            continue

        # Get the source wells for the current item
        source_rows = source_wells_df[source_wells_df['Item'] == item].reset_index(drop=True)

        # Get the destination wells for the current item
        destination_rows = destination_df[['well_name', item]].copy()
        destination_rows.columns = ['Destination Well', 'Transfer Volume']
        destination_rows = destination_rows[destination_rows['Transfer Volume'] > 0].reset_index(drop=True)

        # Create the transfer rows
        source_index = -1
        source_volume_left = source_rows.iloc[source_index]['Volume']

        # Iterate through all destination wells for this item.
        # If there is enough volume in the source well, transfer at once.
        # If not, transfer all remaining volume of the current source well and use the next source well to complete the transfer.
        for _, dest_row in destination_rows.iterrows():
            transfer_volume = dest_row['Transfer Volume']
            while transfer_volume > 0:
                if transfer_volume <= source_volume_left:
                    transfer_table.append({
                        'Item': item,
                        'Source Well': source_rows.iloc[source_index]['Source Well'],
                        'Destination Well': dest_row['Destination Well'],
                        'Transfer Volume': transfer_volume
                    })
                    source_volume_left -= transfer_volume
                    if source_volume_left < minimum_drop_size_nanoliter:
                        source_volume_left = 0
                    transfer_volume = 0
                else:
                    transfer_table.append({
                        'Item': item,
                        'Source Well': source_rows.iloc[source_index]['Source Well'],
                        'Destination Well': dest_row['Destination Well'],
                        'Transfer Volume': source_volume_left
                    })
                    transfer_volume -= source_volume_left
                    source_index += 1
                    if source_index <= len(source_rows):
                        #print(source_index)
                        #print(len(source_rows))
                        source_volume_left = source_rows.iloc[source_index]['Volume']
                    else:
                        #print("helloworld")
                        #print(str(source_index))
                        #print(str(len(source_rows)))
                        raise ValueError(f"Not enough volume available in source wells for item {item}.")

    # Create DataFrame and remove rows with zero transfer volume
    ECHO_transfer_table_df = pd.DataFrame(transfer_table)
    ECHO_transfer_table_df = ECHO_transfer_table_df[ECHO_transfer_table_df['Transfer Volume'] > 0]
    ECHO_transfer_table_df = ECHO_transfer_table_df.reset_index(drop=True)

    # Optionally delete rows corresponding to items in fixed_parts and summarize volumes for each destination well
    if Master_Mix_for_fixed_parts and fixed_parts:
      # Create a new dictionary by multiplying each value in fixed_parts by final_reaction_volume_nanoliter
      fixed_parts_volumes = {key: value * final_reaction_volume_nanoliter for key, value in fixed_parts.items()}



    return ECHO_transfer_table_df, fixed_parts_volumes

ECHO_input_table, fixed_parts_volumes = create_ECHO_transfer_table(destination_df, source_wells_df, final_reaction_volume_nanoliter=final_reaction_volume_nanoliter, minimum_drop_size_nanoliter=minimum_drop_size_nanoliter, fixed_parts=fixed_parts, Master_Mix_for_fixed_parts=Master_Mix_for_fixed_parts)
ECHO_input_table

### 9.5 Export the ECHO input file


In [ ]:
# Creates a csv file that can directly be feeded to an ECHO liquid handler.
ECHO_input_table.to_csv(f'Echo_Input_day{which_day}.csv', index=False)
print("ECHO input file generated. Please check the folder.")

if Master_Mix_for_fixed_parts:
  print("\n" "PLEASE remember to add following volumes in nano litter manually. This is your fixed_part.")
  print(fixed_parts_volumes)

### 9.6 Generate source-well pipetting instructions


In [ ]:
# This cell generates a table instruction for source well pipetting.
source_wells_instruction = source_pipetting_instructions(source_wells_df, fixed_parts, Master_Mix_for_fixed_parts)
source_wells_instruction

In [ ]:
# Download the source well pipetting instruction as a CSV file.
source_wells_instruction.to_csv(f'source_wells_instruction{which_day}.csv', index=False)

## 10. Diagnostics — K most informative combinations

Identifies the K-combination subset of the training set that, when used for training, best predicts the held-out combinations (Spearman R²). Useful for retrospective analysis of which experimental conditions carried the most information.


In [ ]:
k = 20
number_try = 2000

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day + 1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)


aggregated_data_m

In [ ]:
# Create the grid search parameter and scoring functions
param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

# create the grid search object
grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

print('RandomSearchCV ...')
grid.fit(aggregated_data_m.values, aggregated_label_m.values)
results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
print('RandomSearchCV Done!')

In [ ]:
indexs = []
num = 0
while num < number_try:
    numbers = set()
    while len(numbers) < k:
        numbers.add(random.randint(0, len(aggregated_data_m)-1))
    indexs.append(numbers)
    num += 1

In [ ]:
performance = []
best_param = results.params.iloc[0,]
all_set = set(range(len(aggregated_data_m)))
counter = 0
for index in indexs:
    counter += 1
    if counter%100 == 0:print(counter)
    index_test = all_set - index
    index_test = list(index_test)
    index = list(index)
    model = XGBRegressor(objective = 'reg:squarederror', **best_param)
    model.fit(aggregated_data_m.iloc[index].values, aggregated_label_m.iloc[index].values)
    performance.append(spearmanr(model.predict(aggregated_data_m.iloc[index_test].values), aggregated_label_m.iloc[index_test].values).correlation**2)

performance = np.array(performance)

In [ ]:
number_to_select = 5
ind = np.argpartition(performance, -number_to_select)[-number_to_select:]
performance[ind]

In [ ]:
np.array(indexs)[ind]

In [ ]:
pd.DataFrame(np.array(indexs)[ind]).to_csv('K_Most_Informative_Combinations.csv', index=False)

## 11. Diagnostics — feature importances across rounds

For each completed round, refits an XGBoost regressor on the cumulative data and reports the per-feature gain-based importance, then also reports the mutual-information-based importance from `sklearn.feature_selection.mutual_info_regression`.


In [ ]:
n_iter = 200

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

In [ ]:
importance_list = []
print('This may takes minutes!')

for day_i in range(1, day+1):
    aggregated_data_m = pd.DataFrame(columns=desired_cols)
    aggregated_label_m = pd.DataFrame(columns=['yield'])

    for num in range(start_day, day_i + 1):
        if start_day or num>0:
            data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
        else:
            data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

        aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
        aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

    # Create the grid search parameter and scoring functions
    param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

    # creat estimator
    model = XGBRegressor(objective = 'reg:squarederror')

    # create the grid search object
    grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

    print('RandomSearchCV ...', day_i)
    grid.fit(aggregated_data_m.values, aggregated_label_m.values)
    results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
    print('RandomSearchCV Done!')

    model = XGBRegressor(objective = 'reg:squarederror', **results.params.iloc[0,])
    model.fit(aggregated_data_m.values, aggregated_label_m.values)
    importance_list.append(model.feature_importances_)

In [ ]:
importance_df = pd.DataFrame(importance_list, columns=aggregated_data_m.columns)
importance_df

In [ ]:
importance_df.to_csv('importance_df.csv', index=False)

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler_x = StandardScaler()
X = scaler_x.fit_transform(aggregated_data_m.values)
scaler_y = StandardScaler()
y = scaler_y.fit_transform(aggregated_label_m.values)

In [ ]:
importance_df_sklearn = pd.DataFrame(mutual_info_regression(X, y)).T
importance_df_sklearn.columns = aggregated_data_m.columns
importance_df_sklearn

In [ ]:
importance_df_sklearn.to_csv('importance_df_sklearn.csv', index=False)


In [ ]:
import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 2.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 1.0 #set the value globally

fig, ax = plt.subplots()

#ax = sns.barplot(data=importance_df, color='#818689')
ax = sns.barplot(data=importance_df.iloc[[len(importance_df)-1]], color='#818689')
#ax = plt.plot(list(importance_df.iloc[[len(importance_df)-1]].values[0]), marker='o', markersize=10, ls='', color='#F2B342', label='Last Round')

fig.set_size_inches(30, 7)
plt.legend()
plt.xticks(rotation=90)

## 12. Diagnostics — pairwise non-linear interactions

For every pair of design variables, fits a linear model with an added product term and reports the gain in Pearson R relative to the baseline linear fit. Variables whose product term substantially improves the fit are candidates for non-linear interaction.


In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day + 1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)


aggregated_data_m

In [ ]:
from sklearn.linear_model import LinearRegression
from itertools import combinations

model = LinearRegression()
model.fit(aggregated_data_m.values, aggregated_label_m.values)
baseline_score = pearsonr(pd.DataFrame(model.predict(aggregated_data_m.values))[0] ,aggregated_label_m['yield'])
baseline_score

In [ ]:
interaction_scores = {}
for i in combinations(aggregated_data_m.columns, 2):
    X = pd.concat([aggregated_data_m, (aggregated_data_m[i[0]] * aggregated_data_m[i[1]])], axis=1)
    y = aggregated_label_m

    model = LinearRegression()
    model.fit(X.values, y.values)
    score = pearsonr(pd.DataFrame(model.predict(X.values))[0] ,y['yield'])

    interaction_scores[i] = score

In [ ]:
interactions_df = pd.DataFrame(data=np.zeros([len(aggregated_data_m.columns), len(aggregated_data_m.columns)]), columns=aggregated_data_m.columns, index=aggregated_data_m.columns)

for columns, value in interaction_scores.items():
    diff = value[0] - baseline_score[0]
    interactions_df.loc[columns[1], columns[0]] = diff

In [ ]:
corr = np.corrcoef(np.random.randn(len(aggregated_data_m.columns), 200))
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True

fig, ax = plt.subplots(figsize = (21,15))
ax = sns.heatmap(interactions_df, mask=mask, cmap='YlGnBu')

In [ ]:
fig.savefig('Interactions.png', format='png', dpi=1200)
fig.savefig('Interactions.svg', format='svg', dpi=1200)

## 13. Optional downstream analysis (legacy)

> **Note:** the cells below were developed for an earlier cell-free protein synthesis (CFPS) campaign and load auxiliary files (`Yield_all_data.csv`, `Efficiency_all_data.csv`) that are **not produced by the MEVIS in-vivo workflow above**. They are retained here for users who want to perform similar pairplot / top-decile / heatmap analyses on their own datasets — adapt the file paths, column names, and `control_pos` dictionary to your own experimental setup before running. The cells will fail if these files are not present in the working directory.


In [ ]:
%matplotlib inline
sns.set()
sns.set_style("ticks")

In [ ]:
yield_all = pd.read_csv('Yield_all_data.csv')
yield_all.head()

In [ ]:
yield_all=yield_all.drop('yield_std', axis=1)
yield_all.head()

In [ ]:
yield_all.info()

In [ ]:
yield_all.describe()

In [ ]:
plt.figure(figsize=(60, 60))
sns.set_context(rc={"axes.labelsize":24, "lines.linewidth": 20})

yield_pairplot=sns.pairplot(data=yield_all, corner=True, hue='yield', palette='Blues', plot_kws=dict(s=80, edgecolor="none", alpha=0.5))

In [ ]:
#yield_pairplot.savefig('yield_pairplot.png', format='png', dpi=1200)
yield_pairplot.savefig('yield_pairplot.svg', format='svg', dpi=1200)

**Analysis of the top 10% yileds**

In [ ]:
yield_topTenPercent=yield_all.loc[yield_all['yield'] >= yield_all['yield'].quantile(q=0.9)]

In [ ]:
yield_topTenPercent.info()

In [ ]:
yield_topTenPercent=yield_topTenPercent.drop('yield', axis=1)
yield_topTenPercent.head()

In [ ]:
control_pos= {'HEPES':100.0, 'HEPES_pH7.0':1, 'HEPES_pH7.2':1, 'HEPES_pH7.4':1, 'HEPES_pH7.6':1, 'HEPES_pH7.8':1, 'HEPES_pH8.0':1, 'MgCl2':5.0, 'CP':20.0, 'Bicarbonate':50.0, 'Formate':20.0, 'Hbs':5.33925, 'Hbd':0.7272, 'Ecm':0.5765,'Mco':21.9016, 'Mch':0.2836, 'Mcl':2.786,
       'CoA':0.5, 'B12':0.1, 'NADPH':5.0, 'ATP':2.0, 'Pco':3.0616, 'Ccr':0.6167, 'Epi':0.7445, 'Mcm':0.30415, 'Scr':2.615475, 'Ssr':0.552, 'KatE':1.6413, 'Fdh':14.56,
           'CA':0.016675, 'Gor':1.1047, 'CK':0.39215}

yield_topTenPercent = yield_topTenPercent.append(control_pos, ignore_index=True)
yield_topTenPercent.info()
yield_topTenPercent.tail()



In [ ]:

yield_topNorm=yield_topTenPercent.loc[:,:] = yield_topTenPercent.loc[:,:].div(yield_topTenPercent.iloc[-1][:])
yield_topNorm.tail()

In [ ]:
yield_topNormLog=np.log10(yield_topNorm)
yield_topNormLog.tail()

In [ ]:
plt.figure(figsize=(12, 12))
yield_topten_heatmap=sns.heatmap(data=yield_topNormLog, cmap="YlGnBu", linewidths=1, linecolor='black', )

In [ ]:
figure = yield_topten_heatmap.get_figure()
#figure.savefig('yield_topten_heatmap.png', format='png', dpi=1200)
figure.savefig('yield_topten_heatmap.svg', format='svg', dpi=1200)

In [ ]:
from sklearn import preprocessing

scaler=preprocessing.MinMaxScaler()
scaled_yields=scaler.fit_transform(yield_all.sort_values(by='yield'))

plt.figure(figsize=(12, 36))
yield_all_heatmap=sns.heatmap(data=scaled_yields, cmap="YlGnBu", xticklabels=True, yticklabels=False)

In [ ]:
figure = yield_all_heatmap.get_figure()
#figure.savefig('yield_all_heatmap.png', format='png', dpi=1200)
figure.savefig('yield_all_heatmap.svg', format='svg', dpi=1200)

**Efficiency data analysis**

In [ ]:
efficiency_all = pd.read_csv('Efficiency_all_data.csv')
efficiency_all.head()

In [ ]:
efficiency_all=efficiency_all.drop('yield_std', axis=1)
efficiency_all.head()

In [ ]:
plt.figure(figsize=(60, 60))
sns.set_context(rc={"axes.labelsize":24, "lines.linewidth": 20})

efficiency_pairplot=sns.pairplot(data=efficiency_all, corner=True, hue='yield', palette='Blues', plot_kws=dict(s=80, edgecolor="none", alpha=0.5))

In [ ]:
#efficiency_pairplot.savefig('efficiency_pairplot.png', format='png', dpi=1200)
efficiency_pairplot.savefig('efficiency_pairplot.svg', format='svg', dpi=1200)

**Analysis of the top 10% yileds**

In [ ]:
efficiency_topTenPercent=efficiency_all.loc[efficiency_all['yield'] >= efficiency_all['yield'].quantile(q=0.9)]

In [ ]:
efficiency_topTenPercent.info()

In [ ]:
efficiency_topTenPercent=efficiency_topTenPercent.drop('yield', axis=1)
efficiency_topTenPercent.head()

In [ ]:
control_pos= {'HEPES':100.0, 'HEPES_pH7.0':1, 'HEPES_pH7.2':1, 'HEPES_pH7.4':1, 'HEPES_pH7.6':1, 'HEPES_pH7.8':1, 'HEPES_pH8.0':1, 'MgCl2':5.0, 'CP':20.0, 'Bicarbonate':50.0, 'Formate':20.0, 'Hbs':5.33925, 'Hbd':0.7272, 'Ecm':0.5765,'Mco':21.9016, 'Mch':0.2836, 'Mcl':2.786,
       'CoA':0.5, 'B12':0.1, 'NADPH':5.0, 'ATP':2.0, 'Pco':3.0616, 'Ccr':0.6167, 'Epi':0.7445, 'Mcm':0.30415, 'Scr':2.615475, 'Ssr':0.552, 'KatE':1.6413, 'Fdh':14.56,
           'CA':0.016675, 'Gor':1.1047, 'CK':0.39215}

efficiency_topTenPercent = efficiency_topTenPercent.append(control_pos, ignore_index=True)
efficiency_topTenPercent.info()
efficiency_topTenPercent.tail()



In [ ]:

efficiency_topNorm=efficiency_topTenPercent.loc[:,:] = efficiency_topTenPercent.loc[:,:].div(efficiency_topTenPercent.iloc[-1][:])
efficiency_topNorm.tail()

In [ ]:
efficiency_topNormLog=np.log10(efficiency_topNorm)
efficiency_topNormLog.tail()

In [ ]:
plt.figure(figsize=(12, 12))
efficiency_topten_heatmap=sns.heatmap(data=efficiency_topNormLog, cmap="YlGnBu", linewidths=1, linecolor='black', )

In [ ]:
figure = efficiency_topten_heatmap.get_figure()
#figure.savefig('efficiency_topten_heatmap.png', format='png', dpi=1200)
figure.savefig('efficiency_topten_heatmap.svg', format='svg', dpi=1200)

In [ ]:
from sklearn import preprocessing

scaler=preprocessing.MinMaxScaler()
scaled_efficiency=scaler.fit_transform(efficiency_all.sort_values(by='yield'))

plt.figure(figsize=(12, 36))
efficiency_all_heatmap=sns.heatmap(data=scaled_efficiency, cmap="YlGnBu", xticklabels=True, yticklabels=False)

In [ ]:
figure = efficiency_all_heatmap.get_figure()
#figure.savefig('efficiency_all_heatmap.png', format='png', dpi=1200)
figure.savefig('efficiency_all_heatmap.svg', format='svg', dpi=1200)